### Embedding BERT

In [1]:
# Imports
from transformers import BertTokenizer

# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Sample text for tokenization
text = "Embedding models are so cool!"

# Step 1: Tokenize the text
tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
# View
tokens

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': tensor([[ 101, 7861, 8270, 4667, 4275, 2024, 2061, 4658,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

### Embedding Vector DB

In [3]:
!pip install qdrant-client --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 18.0 MB/s eta 0:00:00


In [4]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

# 1. Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# 2. Initialize Qdrant client
client = QdrantClient(":memory:")

# 3. Create embeddings
docs = ["refund policy", "pricing details", "account cancellation"]
vectors = model.encode(docs).tolist()

# 4. Store Vectors
client.create_collection(
    collection_name="my_collection",
    vectors_config = models.VectorParams(size=384,
                                         distance= models.Distance.COSINE)
)

client.upload_collection(collection_name="my_collection",
                         vectors= vectors,
                         payload= [{"source": docs[i]} for i in range(len(docs))])




# 5. Search
query_vector = model.encode("How do I cancel my subscription")

# Result
result = client.query_points(collection_name= 'my_collection',
                             query= query_vector,
                             limit=2,
                             with_payload=True)

print("\n\n ======= RESULTS =========")
result.points


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




 ======= RESULTS =========


[ScoredPoint(id='e6d22e86-736b-4cdd-b30c-6f7368930b29', version=0, score=0.6616353073200185, payload={'source': 'account cancellation'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='913a8d3a-dba8-4830-9f71-a069ff70317d', version=0, score=0.2760082702501182, payload={'source': 'refund policy'}, vector=None, shard_key=None, order_value=None)]

## Fine Tuning

#### Model Out-Of-The-Box

In [24]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers import util

# 1. Load a pre-trained base model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Define your test cases
query = "Brand A Cola Soda"
choices = [
    "Brand B Cola Soda",   # The 'Positive' (Should be closer now)
    "Brand AA Cola Zero Sugar"   # The 'Negative' (Should be further away now)
]

# 2. Encode the text into vectors
query_vec = model.encode(query)
choice_vecs = model.encode(choices)

# 3. Compute Cosine Similarity
# util.cos_sim returns a matrix, so we convert to a list for readability
cos_scores = util.cos_sim(query_vec, choice_vecs)[0].tolist()

print(f"\n\n ======= Results for: {query} ===============")
for i, score in enumerate(cos_scores):
    print(f"-> {choices[i]}: {score:.5f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




 ======= Results for: Brand A Cola Soda ===============
-> Brand B Cola Soda: 0.86003
-> Brand AA Cola Zero Sugar: 0.64906


#### Fine Tuned Model

In [23]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers import util

# 1. Load a pre-trained base model
fine_tuned_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define your training data (Anchors, Positives, and Negatives)
train_examples = [
    InputExample(texts=['Brand A Cola Soda', 'Brand B Cola Soda', 'Brand AA Cola Zero Sugar'])
]

# 3. Create a DataLoader and choose a Loss Function
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.TripletLoss(model=fine_tuned_model)

# 4. Tune the model
fine_tuned_model.fit(train_objectives=[(train_dataloader, train_loss)],
                     optimizer_params={'lr': 9e-5},
                     epochs=20)


# 1. Define your test cases
query = "Brand A Cola Soda"
choices = [
    "Brand B Cola Soda",   # The 'Positive' (Should be closer now)
    "Brand AA Cola Zero Sugar"   # The 'Negative' (Should be further away now)
]

# 2. Encode the text into vectors
query_vec = fine_tuned_model.encode(query)
choice_vecs = fine_tuned_model.encode(choices)

# 3. Compute Cosine Similarity
# util.cos_sim returns a matrix, so we convert to a list for readability
cos_scores = util.cos_sim(query_vec, choice_vecs)[0].tolist()

print(f"\n\n ======== Results for: {query} ====================")
for i, score in enumerate(cos_scores):
    print(f"-> {choices[i]}: {score:.5f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss




 ======== Results for: Brand A Cola Soda ====================
-> Brand B Cola Soda: 0.85773
-> Brand AA Cola Zero Sugar: 0.65294
